# Phase 2.6 — Confirmatory unseen-family S3 replication

This notebook runs the frozen unseen-family replication. The predictor is fixed before model collection and is not refit on this data.


## 1. Recover `/content` safely


In [ ]:
import os, shutil, subprocess
os.chdir('/content')
repo = '/content/proof-path-invariance'
if os.path.exists(repo):
    shutil.rmtree(repo)
print('cwd:', os.getcwd())


## 2. Clone the latest repository


In [ ]:
subprocess.run([
    'git','clone',
    'https://github.com/Kairose-master/proof-path-invariance.git',
    repo
], check=True, cwd='/content')
os.chdir(repo)
print('cwd:', os.getcwd())


## 3. Install runner dependencies


In [ ]:
!python3 -m pip install -q -r requirements-runner.txt


## 4. Generate the frozen unseen-family benchmark


In [ ]:
!python3 scripts/generate_s3_unseen_benchmark.py --out /tmp/s3_unseen_v0.jsonl


## 5. Validate 128 cases / 768 prompts


In [ ]:
!python3 scripts/validate_s3_unseen_benchmark.py /tmp/s3_unseen_v0.jsonl


## 6. Verify the frozen benchmark hash


In [ ]:
!python3 scripts/verify_s3_unseen_lock.py /tmp/s3_unseen_v0.jsonl


Continue only if the final line is `unseen-family benchmark lock verified`.


Continue only if validation succeeds. If the repository includes a frozen unseen-family lock verifier, run it before the model cell as well.


## 7. Run Pythia-70M on all 768 prompts


In [ ]:
!mkdir -p results
!python3 scripts/run_hf_s3_margin.py --prompts /tmp/s3_unseen_v0.jsonl --out results/pythia70m-step143000-s3-unseen-v0.jsonl --run-id pythia70m-step143000-s3-unseen-v0


The raw result is exclusive-create. Do not rerun this cell against the same output path.


## 8. **Download RAW DATA immediately**

Run this before scoring or closing Colab. This raw JSONL is the primary experiment artifact.


In [ ]:
from google.colab import files
import os
RAW = '/content/proof-path-invariance/results/pythia70m-step143000-s3-unseen-v0.jsonl'
if not os.path.exists(RAW):
    raise FileNotFoundError(RAW)
print('Downloading RAW DATA:', RAW)
files.download(RAW)


Expected filename: `pythia70m-step143000-s3-unseen-v0.jsonl`.


## 9. Run the frozen confirmatory scorer


In [ ]:
!python3 scripts/score_s3_unseen_confirmatory.py results/pythia70m-step143000-s3-unseen-v0.jsonl | tee /content/phase2_6_confirmatory_score.json


Primary field: `primary_endpoint.result.skill_vs_zero`. Confirmatory success is exactly `skill_vs_zero > 0`.


## 10. Download scorer JSON


In [ ]:
files.download('/content/phase2_6_confirmatory_score.json')


Do not close Colab until both the raw JSONL and scorer JSON have downloaded.
